1. Define the State — the typed object that every node reads and writes:

In [1]:
from langgraph.graph import StateGraph, MessagesState

# MessagesState is a built-in: just a list of messages + add_messages reducer
graph = StateGraph(MessagesState)

In [2]:
print(graph)

2. Define nodes — plain Python functions that take State and return a State update:

In [24]:
import os
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode
from langchain_community.tools import DuckDuckGoSearchResults

# 1. Load Environment Variables
load_dotenv()

# 2. Define Sample Tools
@tool
def search_web(query: str) -> str:
    """Search the web for real-time information."""
    # 1. Initialize the tool instance
    search_tool = DuckDuckGoSearchResults()
    
    # 2. Execute the search using the .invoke() method
    results = search_tool.invoke(query)
    
    return results

tools_list = [search_web]

# 3. Instantiate ToolNode (Handles all tool executions autonomously)
tool_node = ToolNode(tools_list)

# 4. Initialize LLM and Bind Tools
llm = ChatOpenAI(
    model="gpt-5-mini",  
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL")
).bind_tools(tools_list)

# 5. Define Graph State
class AgentState(MessagesState):
    pass

# 6. Define Agent Node Function
def agent(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# 7. Route Logic (Decides to loop to tools or finish)
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 8. Build and Compile the Graph
workflow = StateGraph(AgentState)

workflow.add_node("agent", agent)

workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

app = workflow.compile()

# 9. Test the Agent
if __name__ == "__main__":
    inputs = {"messages": [HumanMessage(content="What is the weather like in Ahmedabad today? Give me an answer ")]}
    for output in app.stream(inputs):
        print(output)


{'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 140, 'total_tokens': 204, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini', 'system_fingerprint': None, 'id': 'chatcmpl-DjKhD7McpItfRViNHoOKMHVdIjqzH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e5e30-5589-7371-9887-05bc61310585-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'Ahmedabad weather today'}, 'id': 'call_xJsp5zPuKRRZh7gGMK32hIL0', 'type': 'tool_call'}, {'name': 'search_web', 'args': {'query': 'Ahmedabad current temperature weather forecast now'}, 'id': 'call_fvDrQtltyYHfNClMZqcLMhhi', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'

In [31]:
from langgraph.prebuilt import tools_condition
import os
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode
from langchain_community.tools import DuckDuckGoSearchResults

# 1. Load Environment Variables
load_dotenv()

# 2. Define Sample Tools
@tool
def search_web(query: str) -> str:
    """Search the web for real-time information."""
    # 1. Initialize the tool instance
    search_tool = DuckDuckGoSearchResults()
    
    # 2. Execute the search using the .invoke() method
    results = search_tool.invoke(query)
    
    return results

tools_list = [search_web]

# 3. Instantiate ToolNode (Handles all tool executions autonomously)
tool_node = ToolNode(tools_list)

# 4. Initialize LLM and Bind Tools
llm = ChatOpenAI(
    model="gpt-5-mini",  
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL")
).bind_tools(tools_list)

# 5. Define Graph State
class AgentState(MessagesState):
    pass

# 6. Define Agent Node Function
def agent(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# 7. Route Logic (Decides to loop to tools or finish)
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 8. Build and Compile the Graph
graph = StateGraph(AgentState)
graph.add_node("agent", agent)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")
graph.add_conditional_edges("agent", tools_condition)  # → tools OR __end__
graph.add_edge("tools", "agent")                       # always loop back

app = graph.compile()
result = app.invoke({"messages": [("user", "What's 15% of current Apple's market cap?")]})
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's 15% of current Apple's market cap?
================================== Ai Message ==================================
Tool Calls:
  search_web (call_ElF90tEdd8L7R5by9vR2JSod)
 Call ID: call_ElF90tEdd8L7R5by9vR2JSod
  Args:
    query: Apple market capitalization current market cap AAPL market cap today
================================= Tool Message =================================
Name: search_web

snippet: April 23, 2026 - Current and historical market capitalization for Apple Inc. (AAPL) stock, including annual, quarterly and daily history with a chart and statistics., title: Apple (AAPL) Market Cap & Net Worth, link: https://stockanalysis.com/stocks/aapl/market-cap/, snippet: 3 weeks ago - Track Apple Inc stock price on the chart and check out the list of the most volatile stocks — is Apple Inc there? ... Today Apple Inc has the market capitalization of 4.54 T, it has increased by 2.58% over the l